Libraries

In [17]:
pip install lasio

Defaulting to user installation because normal site-packages is not writeable
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [41]:
import lasio
import pandas as pd
from pathlib import Path

LAS Reader Function

In [42]:
def read_las_file_flexible(file_path):
    """
    Reads LAS content regardless of extension.
    Accepts .las, .txt, or extensionless files.
    """
    file_path = clean_path(file_path)
    
    if not file_path.exists():
        raise FileNotFoundError(f"❌ File not found: {file_path}")
    
    try:
        # lasio reads purely by content, not extension
        las = lasio.read(file_path, ignore_header_errors=True)
    except Exception as e:
        raise ValueError(f"❌ File is not valid LAS content: {file_path}\n{e}")
    
    curve_names = [curve.mnemonic for curve in las.curves]
    curve_units = [curve.unit for curve in las.curves]
    data_df = las.df()
    
    return curve_names, curve_units, data_df


CSV Formatter Function

In [43]:
def build_csv_dataframe(curve_names, curve_units, data_df):
    """
    Builds final CSV dataframe:
    Row 1: Feature names
    Row 2: Units
    Row 3+: Data
    """
    
    # Convert dataframe to list of lists
    data_values = data_df.reset_index().values.tolist()
    
    # Insert header rows
    final_table = []
    final_table.append(curve_names)
    final_table.append(curve_units)
    final_table.extend(data_values)
    
    # Convert to pandas DataFrame
    final_df = pd.DataFrame(final_table)
    
    return final_df


Single LAS to CSV converter

In [44]:
def convert_single_las_to_csv(file_path, output_directory=None):
    
    file_path = clean_path(file_path)
    
    if output_directory:
        output_directory = Path(output_directory)
        output_directory.mkdir(parents=True, exist_ok=True)
        output_csv_path = output_directory / (file_path.stem + ".csv")
    else:
        output_csv_path = file_path.with_suffix(".csv")
    
    print(f"Processing: {file_path.name}")
    
    curve_names, curve_units, data_df = read_las_file_flexible(file_path)
    
    final_df = build_csv_dataframe(curve_names, curve_units, data_df)
    
    final_df.to_csv(output_csv_path, index=False, header=False)
    
    print(f"✅ Saved CSV → {output_csv_path}")
    
    return output_csv_path


Batch LAS to CSV converter

In [45]:
def convert_batch_las_to_csv(parent_directory, output_directory=None):
    """
    Converts all LAS-content files:
    .las, .txt, or extensionless
    """
    parent_directory = Path(parent_directory)
    
    # Grab everything that looks like a file
    files = [f for f in parent_directory.iterdir() if f.is_file()]
    
    if not files:
        print("❌ No files found!")
        return
    
    print(f"Found {len(files)} file(s). Trying LAS parsing...\n")
    
    success = 0
    
    for i, file in enumerate(files, 1):
        print(f"[{i}/{len(files)}] {file.name}")
        try:
            convert_single_las_to_csv(file, output_directory)
            success += 1
        except:
            print("⚠️ Skipped — not LAS content\n")
    
    print(f"🎉 Completed! {success} LAS file(s) converted.")


Single file execution

In [47]:
# Set LAS file path
las_path = "/Users/apple/Downloads/welldata/ROTTERDAM-08-SIDETRACK1/logdocuments/drill_mech_depth_30jul20.txt"

# Convert single LAS to CSV
convert_single_las_to_csv(las_path)


Processing: drill_mech_depth_30jul20.txt
✅ Saved CSV → /Users/apple/Downloads/welldata/ROTTERDAM-08-SIDETRACK1/logdocuments/drill_mech_depth_30jul20.csv


PosixPath('/Users/apple/Downloads/welldata/ROTTERDAM-08-SIDETRACK1/logdocuments/drill_mech_depth_30jul20.csv')